# Laboratorio 2 · Regressione e discesa del gradiente

**ITS Academy Cremona** · Corso Intelligenza Artificiale e Machine Learning · Digitalizzazione dei processi industriali

Lezione 2 (ore 3–4) · Regressione: come impara una macchina · Docente: Rosario Marra

**Durata:** 45 minuti · **Modalità:** A coppie

**Nome e cognome:** _scrivi qui_  
**Compagno/a di lavoro (se in coppia):** _scrivi qui_

## Obiettivo

Addestrare e valutare una regressione lineare, riconoscere l’overfitting e osservare l’effetto del learning rate nella discesa del gradiente.

## Prima di iniziare

1. **File › Salva una copia su Drive**: lavora sulla tua copia.
2. Esegui le celle **in ordine**, dall’alto in basso (Maiusc + Invio).
3. Leggi ogni output: le domande finali si basano su quello che vedi.

## I passi

1. **Carica i dati.** Esegui la cella di caricamento del dataset AI4I.
2. **Una regressione semplice.** Prevedi la coppia (Torque) dalla velocità di rotazione e disegna punti e retta.
3. **Leggi le metriche.** Calcola MAE, RMSE e R² sul test e scrivi a parole quanto sbaglia il modello, in Nm.
4. **Provoca l’overfitting.** Con solo 30 righe di training confronta polinomi di grado 1, 3 e 15: errore su training e su test.
5. **Discesa del gradiente a mano.** Esegui l’algoritmo in NumPy con learning rate 0,0001 · 0,01 · 0,05 e confronta le curve dell’errore.
6. **Conclusioni.** Quale modello e quale learning rate sceglieresti? Perché?

## 1. Caricamento

In [ ]:
!pip install ucimlrepo -q
from ucimlrepo import fetch_ucirepo
import pandas as pd, numpy as np, matplotlib.pyplot as plt

ai4i = fetch_ucirepo(id=601)                          # AI4I 2020 Predictive Maintenance (UCI)
df = ai4i.data.original.copy()                        # tabella completa: 10.000 righe
df.columns = [c.split(" [")[0] for c in df.columns]   # toglie le unità dai nomi delle colonne
df.head()

## 2. Regressione lineare

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = df[["Rotational speed"]]; y = df["Torque"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
modello = LinearRegression().fit(X_tr, y_tr)
pred = modello.predict(X_te)
print("pendenza a =", modello.coef_[0], "  intercetta c =", modello.intercept_)
print("MAE :", mean_absolute_error(y_te, pred))
print("RMSE:", mean_squared_error(y_te, pred) ** 0.5)
print("R2  :", r2_score(y_te, pred))

plt.scatter(X_te, y_te, s=3, alpha=.4); plt.plot(X_te, pred, color="red")
plt.xlabel("Velocità (rpm)"); plt.ylabel("Coppia (Nm)"); plt.show()

## 3. Overfitting con i polinomi

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

piccolo = X_tr.sample(30, random_state=1); y_piccolo = y_tr.loc[piccolo.index]
for grado in [1, 3, 15]:
    m = make_pipeline(StandardScaler(), PolynomialFeatures(grado), LinearRegression())
    m.fit(piccolo, y_piccolo)
    e_tr = mean_squared_error(y_piccolo, m.predict(piccolo)) ** 0.5
    e_te = mean_squared_error(y_te, m.predict(X_te)) ** 0.5
    print(f"grado {grado:2d}   RMSE training {e_tr:7.2f}   RMSE test {e_te:10.2f}")

## 4. Discesa del gradiente in NumPy

In [ ]:
x = np.linspace(0, 10, 50); y_v = 3 * x + 2 + np.random.randn(50)   # dati: y ≈ 3x + 2

def discesa(lr, epoche=200):
    w, b, storia = 0.0, 0.0, []
    for _ in range(epoche):
        errore = (w * x + b) - y_v
        storia.append(np.mean(errore ** 2))
        w -= lr * 2 * np.mean(errore * x)     # passo opposto al gradiente rispetto a w
        b -= lr * 2 * np.mean(errore)         # passo opposto al gradiente rispetto a b
    return w, b, storia

for lr in [0.0001, 0.01, 0.05]:
    w, b, storia = discesa(lr)
    plt.plot(storia, label=f"lr = {lr}   w={w:.2f}  b={b:.2f}")
plt.yscale("log"); plt.xlabel("epoca"); plt.ylabel("errore (MSE)"); plt.legend(); plt.show()

## Le tue risposte

Rispondi in modo breve ma motivato, citando i numeri che hai ottenuto.

**Domanda 1.** Quanto sbaglia in media la regressione lineare sul test (MAE, in Nm)? R² è buono?

✍️ **La tua risposta:** _scrivi qui_

**Domanda 2.** Nella tabella dei polinomi, quale grado va in overfitting? Da quali numeri lo capisci?

✍️ **La tua risposta:** _scrivi qui_

**Domanda 3.** Cosa succede all’errore con learning rate 0,0001, 0,01 e 0,05?

✍️ **La tua risposta:** _scrivi qui_

**Domanda 4.** Quale modello porteresti in produzione e perché?

✍️ **La tua risposta:** _scrivi qui_

## Consegna su Classroom

- Il notebook completato in formato .ipynb, oppure il link in sola lettura
- Le risposte alle 4 domande nelle celle «La tua risposta»

Per scaricare: **File › Scarica › Scarica .ipynb**, poi in Classroom **Aggiungi o crea › File** e **Consegna**.

### Griglia di valutazione

| Criterio | Punti |
| --- | --- |
| Regressione addestrata e valutata sul test | 3 |
| Overfitting riconosciuto e spiegato | 3 |
| Effetto del learning rate descritto correttamente | 3 |
| Scelta finale motivata | 1 |
| **Totale** | **10** |

**Suggerimenti:** Un RMSE si legge sempre nell’unità del target: qui Newton-metro. Se una curva dell’errore sale o «esplode», il learning rate è troppo alto. Facoltativo: su playground.tensorflow.org prova lo stesso esperimento con il dataset «Circle».